In [ ]:
#!/usr/bin/env python3
"""
Verifies a local backup (from download_backup.py) still matches what the
server/Cloudinary currently has - and tells you clearly *why* when it
doesn't, rather than just pass/fail.

For every row in submissions.csv with an audioFile:
  1. Confirms the local .wav exists and its size matches what was recorded
     at download time (catches a truncated/corrupted download).
  2. Re-fetches that submission live from the server right now and compares:
       - Cloudinary no longer has the audio (already cleaned up) -> expected,
         reported as OK. Your local file is now the only copy that exists.
       - Cloudinary still has it, but the size differs from your local copy
         -> the submission changed (re-recorded) since you backed it up.
       - not found at all -> the project/task was deleted since your backup.

Doesn't touch Cloudinary directly - it verifies against the app's own API,
which already reflects Cloudinary's real state, so no Cloudinary credentials
are needed here.

Requires: pip install requests

Edit BASE_URL / ADMIN_EMAIL / ADMIN_PASSWORD / BACKUP_DIR below, then run:
    python3 verify_backup.py
"""
import csv
import os
import sys
import requests

# ── Edit these, then run ────────────────────────────────────────────────────
BASE_URL = "https://your-app.example.com/api"  # no trailing slash; use
                                                # "http://localhost:5001/api" for local
ADMIN_EMAIL = "admin@example.com"
ADMIN_PASSWORD = "CHANGE_ME"
BACKUP_DIR = "bolochinese_backup"  # the folder download_backup.py wrote into
# ─────────────────────────────────────────────────────────────────────────────

PAGE_SIZE = 100
REQUEST_TIMEOUT = 60

# Cloudinary re-packages the WAV container on upload, so the size it actually
# stores/serves can be a handful of bytes off from the "audioFileSizeBytes"
# recorded in the database at upload time (which was the size of the raw
# buffer *before* Cloudinary touched it) - not corruption. A real truncated
# or corrupted download will be off by far more than this. Only flag a
# difference bigger than the tolerance.
SIZE_TOLERANCE_BYTES = 256


def login(session):
    r = session.post(
        f"{BASE_URL}/auth/login",
        json={"email": ADMIN_EMAIL, "password": ADMIN_PASSWORD},
        timeout=REQUEST_TIMEOUT,
    )
    r.raise_for_status()
    session.headers["Authorization"] = f"Bearer {r.json()['data']['token']}"


def get_projects(session):
    r = session.get(f"{BASE_URL}/admin/projects", timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    return r.json()["data"]


def get_all_submissions(session, project_id):
    """Every submission for a project, fetched fresh right now - includes
    ones whose audio has already been purged (they just show audio: null)."""
    items = []
    page = 1
    while True:
        r = session.get(
            f"{BASE_URL}/admin/projects/{project_id}/submissions",
            params={"page": page, "limit": PAGE_SIZE},
            timeout=REQUEST_TIMEOUT,
        )
        r.raise_for_status()
        data = r.json()["data"]
        items.extend(data["items"])
        if page >= data["pagination"]["totalPages"]:
            break
        page += 1
    return items


def main():
    if ADMIN_PASSWORD == "CHANGE_ME" or "your-app.example.com" in BASE_URL:
        sys.exit("Edit BASE_URL / ADMIN_EMAIL / ADMIN_PASSWORD at the top of this script first.")

    csv_path = os.path.join(BACKUP_DIR, "submissions.csv")
    if not os.path.exists(csv_path):
        sys.exit(f"Can't find {csv_path} - run download_backup.py first.")

    with open(csv_path, newline="", encoding="utf-8-sig") as f:
        rows = list(csv.DictReader(f))

    audio_rows = [r for r in rows if r.get("audioFile")]
    print(f"Checking {len(audio_rows)} audio file(s) from {csv_path}...\n")

    session = requests.Session()
    login(session)

    projects = get_projects(session)
    print(f"Fetching current server state for {len(projects)} project(s)...")

    # (project name, username, dialogueId) -> live submission item. Uses the
    # same "username or empty string" rule download_backup.py used for the
    # CSV column (not the fuller "or email or unknown" one it used for
    # filenames), so the keys line up with what's actually in the CSV.
    live_by_key = {}
    for project in projects:
        for item in get_all_submissions(session, project["_id"]):
            task = item.get("taskId") or {}
            user = item.get("userId") or {}
            dialogue_id = task.get("dialogueId") or item["_id"]
            username = user.get("username") or ""
            live_by_key[(project["name"], username, dialogue_id)] = item

    print("Done fetching. Comparing...\n")

    ok, purged, stale, missing_local, size_mismatch, not_found = [], [], [], [], [], []

    for row in audio_rows:
        label = f"{row['project']}/{row['username']}/{row['dialogueId']}"
        local_path = os.path.join(BACKUP_DIR, row["audioFile"])

        if not os.path.exists(local_path):
            missing_local.append(label)
            print(f"  ✗ MISSING LOCALLY: {label}")
            continue

        local_size = os.path.getsize(local_path)
        recorded_size = int(row.get("audioFileSizeBytes") or 0)
        if recorded_size and abs(local_size - recorded_size) > SIZE_TOLERANCE_BYTES:
            size_mismatch.append(label)
            print(f"  ✗ CORRUPTED DOWNLOAD: {label} (disk has {local_size} bytes, CSV recorded {recorded_size})")
            continue

        key = (row["project"], row["username"], row["dialogueId"])
        live = live_by_key.get(key)
        if live is None:
            not_found.append(label)
            print(f"  ✗ NOT FOUND ON SERVER: {label} (project/task deleted since backup?)")
            continue

        live_audio = live.get("audio") or {}
        if not live_audio.get("url"):
            purged.append(label)
            print(f"  ✓ already cleaned up on Cloudinary: {label} (your local copy is now the only one)")
            continue

        live_size = live_audio.get("fileSizeBytes") or 0
        if live_size and abs(live_size - local_size) > SIZE_TOLERANCE_BYTES:
            stale.append(label)
            print(f"  ⚠ CHANGED SINCE BACKUP: {label} (Cloudinary now has {live_size} bytes, your copy has {local_size})")
            continue

        ok.append(label)
        print(f"  ✓ matches: {label}")

    print(
        f"\nDone. {len(ok)} match, {len(purged)} already cleaned up (local copy is the only one), "
        f"{len(stale)} changed since backup, {len(missing_local)} missing locally, "
        f"{len(size_mismatch)} corrupted download, {len(not_found)} not found on server "
        f"(of {len(audio_rows)} total)."
    )

    problems = missing_local + size_mismatch + not_found + stale
    if problems:
        print(f"\n{len(problems)} item(s) need attention - see the ✗/⚠ lines above.")
        sys.exit(1)
    print("\nEverything checks out.")


if __name__ == "__main__":
    main()